In [1]:
import pickle
import random
import numpy as np

import gymnasium as gym
from gymnasium import spaces
from gymnasium.wrappers import FlattenObservation, TransformAction
from stable_baselines3 import PPO
from stable_baselines3.common.vec_env import SubprocVecEnv, VecMonitor, VecNormalize
from stable_baselines3.common.env_util import make_vec_env
from stable_baselines3.common.env_checker import check_env
from stable_baselines3.common.monitor import Monitor
from stable_baselines3.common.callbacks import BaseCallback

from rl_firm.env import make_sb_env

# Treating unprofitability as true death

In [2]:
class RewardCallback(BaseCallback):
    def __init__(self, verbose=0):
        super().__init__(verbose)
        self.episode_rewards = []
        self.episode_lengths = []
    
    def _on_step(self) -> bool:
        # Check if episode ended
        if self.locals.get('dones', [False])[0]:
            # Get episode info from Monitor wrapper
            info = self.locals.get('infos', [{}])[0]
            if 'episode' in info:
                length = info['episode']['l']
                reward = info['episode']['r']
                self.episode_lengths.append(length)
                self.episode_rewards.append(reward)
                recent_avg_length = np.mean(self.episode_lengths[-20:])
                recent_avg = np.mean(self.episode_rewards[-20:])
                print(f"Episode {len(self.episode_rewards)}: Recent average episodic reward: {recent_avg:.2f}, recent average episodic length: {recent_avg_length:.2f}")
        return True

callback = RewardCallback()

In [3]:
vec_env = make_vec_env(make_sb_env, n_envs=8, vec_env_cls=SubprocVecEnv)
vec_env = VecMonitor(vec_env)

/Users/hieule/Library/Caches/pypoetry/virtualenvs/thesis-5oUIbZFM-py3.13/lib/python3.13/site-packages/gymnasium/spaces/box.py:235: UserWarning: WARN: Box low's precision lowered by casting to float32, current low.dtype=float64
  gym.logger.warn(
/Users/hieule/Library/Caches/pypoetry/virtualenvs/thesis-5oUIbZFM-py3.13/lib/python3.13/site-packages/gymnasium/spaces/box.py:305: UserWarning: WARN: Box high's precision lowered by casting to float32, current high.dtype=float64
  gym.logger.warn(
/Users/hieule/Library/Caches/pypoetry/virtualenvs/thesis-5oUIbZFM-py3.13/lib/python3.13/site-packages/gymnasium/spaces/box.py:235: UserWarning: WARN: Box low's precision lowered by casting to float32, current low.dtype=float64
  gym.logger.warn(
/Users/hieule/Library/Caches/pypoetry/virtualenvs/thesis-5oUIbZFM-py3.13/lib/python3.13/site-packages/gymnasium/spaces/box.py:305: UserWarning: WARN: Box high's precision lowered by casting to float32, current high.dtype=float64
  gym.logger.warn(
/Users/hieul

In [4]:
%%time

# Initialize the PPO model
ppo = PPO(
    policy="MlpPolicy",            
    env=vec_env,                            
    n_steps=1024,
    batch_size=2048,                               
    gamma=0.999,                  
    gae_lambda=0.95,             
    verbose=1,                    
    device="auto"               
)

Using cpu device
CPU times: user 397 ms, sys: 150 ms, total: 547 ms
Wall time: 791 ms


## 500k

In [5]:
%%time

ppo.learn(total_timesteps=500000, callback=callback, reset_num_timesteps=False)

Episode 1: Recent average episodic reward: 0.15, recent average episodic length: 17.00
Episode 2: Recent average episodic reward: 0.20, recent average episodic length: 25.50
Episode 3: Recent average episodic reward: 0.20, recent average episodic length: 23.67
Episode 4: Recent average episodic reward: 0.16, recent average episodic length: 20.50
Episode 5: Recent average episodic reward: 0.13, recent average episodic length: 17.00
Episode 6: Recent average episodic reward: 0.11, recent average episodic length: 15.33
Episode 7: Recent average episodic reward: 0.15, recent average episodic length: 16.14
Episode 8: Recent average episodic reward: 0.15, recent average episodic length: 16.38
Episode 9: Recent average episodic reward: 0.14, recent average episodic length: 16.44
Episode 10: Recent average episodic reward: 0.13, recent average episodic length: 15.80
Episode 11: Recent average episodic reward: 0.12, recent average episodic length: 16.09
Episode 12: Recent average episodic rewar

In [6]:
ppo.save('results/model/true_death/500k')

## 1m

In [7]:
%%time

ppo.learn(total_timesteps=500000, callback=callback, reset_num_timesteps=False)

Episode 769: Recent average episodic reward: 2.55, recent average episodic length: 154.95
Episode 770: Recent average episodic reward: 2.73, recent average episodic length: 157.10
Episode 771: Recent average episodic reward: 2.62, recent average episodic length: 152.50
Episode 772: Recent average episodic reward: 2.66, recent average episodic length: 154.75
Episode 773: Recent average episodic reward: 2.58, recent average episodic length: 153.25
Episode 774: Recent average episodic reward: 2.72, recent average episodic length: 153.50
Episode 775: Recent average episodic reward: 2.76, recent average episodic length: 156.60
Episode 776: Recent average episodic reward: 2.81, recent average episodic length: 153.10
---------------------------------
| rollout/           |          |
|    ep_len_mean     | 137      |
|    ep_rew_mean     | 2.13     |
| time/              |          |
|    fps             | 34       |
|    iterations      | 1        |
|    time_elapsed    | 234      |
|    tot

In [8]:
ppo.save('results/model/true_death/1m')

# 1500k

In [9]:
%%time

ppo.learn(total_timesteps=1000000, callback=callback, reset_num_timesteps=False)

Episode 1975: Recent average episodic reward: 3.58, recent average episodic length: 129.70
Episode 1976: Recent average episodic reward: 3.82, recent average episodic length: 131.90
Episode 1977: Recent average episodic reward: 3.98, recent average episodic length: 137.70
Episode 1978: Recent average episodic reward: 3.82, recent average episodic length: 137.55
Episode 1979: Recent average episodic reward: 3.83, recent average episodic length: 136.10
Episode 1980: Recent average episodic reward: 4.05, recent average episodic length: 138.35
Episode 1981: Recent average episodic reward: 3.98, recent average episodic length: 135.65
---------------------------------
| rollout/           |          |
|    ep_len_mean     | 131      |
|    ep_rew_mean     | 2.43     |
| time/              |          |
|    fps             | 41       |
|    iterations      | 1        |
|    time_elapsed    | 195      |
|    total_timesteps | 1523712  |
---------------------------------
Episode 1982: Recent av

In [8]:
ppo.save('results/model/true_death/1500k')

## 3m

In [10]:
%%time

ppo.learn(total_timesteps=1500000, callback=callback, reset_num_timesteps=False)

Episode 2898: Recent average episodic reward: 2.40, recent average episodic length: 139.45
Episode 2899: Recent average episodic reward: 2.46, recent average episodic length: 139.35
Episode 2900: Recent average episodic reward: 2.32, recent average episodic length: 139.00
Episode 2901: Recent average episodic reward: 2.70, recent average episodic length: 145.05
Episode 2902: Recent average episodic reward: 2.90, recent average episodic length: 146.45
Episode 2903: Recent average episodic reward: 2.70, recent average episodic length: 143.90
---------------------------------
| rollout/           |          |
|    ep_len_mean     | 143      |
|    ep_rew_mean     | 2.81     |
| time/              |          |
|    fps             | 43       |
|    iterations      | 1        |
|    time_elapsed    | 188      |
|    total_timesteps | 2531328  |
---------------------------------
Episode 2904: Recent average episodic reward: 2.67, recent average episodic length: 143.95
Episode 2905: Recent av

In [11]:
ppo.save('results/model/true_death/3m')